# ArogyaVani RAG — Clean & Optimized Prototype

This notebook is a cleaned version of the original RAG prototype. It keeps the same core approach—PDF extraction, multilingual embeddings, FAISS retrieval, Gemini grounded generation, and source metadata—while removing duplicated cells, Colab-specific orchestration, repeated functions, and unnecessary voice/UI code.

**Production direction:** the ingestion/index build step should run separately from the FastAPI query service. The existing ArogyaVani voice API can call the RAG service after STT and before TTS.

In [ ]:
# Install dependencies in a fresh environment/Colab runtime.
!pip -q install -U google-genai sentence-transformers faiss-cpu pypdf sarvamai

In [ ]:
from __future__ import annotations

import json
import os
import re
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Iterable

import numpy as np
from pypdf import PdfReader

# Optional until the real RAG run: these imports should succeed after the install cell.
import faiss
from sentence_transformers import SentenceTransformer
from google import genai

print("Imports ready")

In [ ]:
# Configuration
BASE_DIR = Path.cwd()
PDF_DIR = BASE_DIR / "trusted_pdfs"
ARTIFACT_DIR = BASE_DIR / "rag_artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
GEMINI_MODEL = "gemini-3.5-flash-lite"
CHUNK_SIZE_WORDS = 1000
CHUNK_OVERLAP_WORDS = 150
DEFAULT_TOP_K = 5
DEFAULT_MIN_SCORE: float | None = None  # Set a threshold only after evaluating retrieval on your dataset.

# Put keys in the environment for a real run. Never commit them.
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "").strip()
print({
    "pdf_dir": str(PDF_DIR),
    "artifact_dir": str(ARTIFACT_DIR),
    "embedding_model": EMBEDDING_MODEL,
    "gemini_model": GEMINI_MODEL,
    "gemini_key_present": bool(GEMINI_API_KEY),
})

In [ ]:
@dataclass(frozen=True)
class PageDocument:
    text: str
    source: str
    page: int

@dataclass(frozen=True)
class Chunk:
    text: str
    source: str
    page: int
    chunk_id: int

@dataclass(frozen=True)
class RetrievalResult:
    text: str
    source: str
    page: int
    chunk_id: int
    score: float

def clean_text(text: str) -> str:
    text = re.sub(r"\s+", " ", text or "").strip()
    return text

def load_pdfs(pdf_dir: Path) -> list[PageDocument]:
    docs: list[PageDocument] = []
    for path in sorted(pdf_dir.glob("*.pdf")):
        reader = PdfReader(str(path))
        for page_no, page in enumerate(reader.pages, start=1):
            text = clean_text(page.extract_text() or "")
            if text:
                docs.append(PageDocument(text=text, source=path.name, page=page_no))
    return docs

def chunk_words(text: str, size: int = CHUNK_SIZE_WORDS, overlap: int = CHUNK_OVERLAP_WORDS) -> list[str]:
    words = text.split()
    if not words:
        return []
    if overlap >= size:
        raise ValueError("chunk overlap must be smaller than chunk size")
    out: list[str] = []
    start = 0
    while start < len(words):
        end = min(start + size, len(words))
        out.append(" ".join(words[start:end]))
        if end == len(words):
            break
        start = end - overlap
    return out

def build_chunks(documents: Iterable[PageDocument]) -> list[Chunk]:
    chunks: list[Chunk] = []
    for doc in documents:
        for idx, text in enumerate(chunk_words(doc.text)):
            chunks.append(Chunk(text=text, source=doc.source, page=doc.page, chunk_id=idx))
    return chunks

print("Functions defined")

In [ ]:
# Load and chunk the trusted PDFs.
documents = load_pdfs(PDF_DIR)
chunks = build_chunks(documents)
print(f"Pages loaded: {len(documents)}")
print(f"Chunks created: {len(chunks)}")
print("Sample:", asdict(chunks[0]) if chunks else "No chunks")

In [ ]:
# Load the multilingual embedding model once.
embedding_model = SentenceTransformer(EMBEDDING_MODEL)

def embed_texts(texts: list[str]) -> np.ndarray:
    vectors = embedding_model.encode(
        texts,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    )
    return np.asarray(vectors, dtype=np.float32)

print("Embedding dimension:", embedding_model.get_sentence_embedding_dimension())

In [ ]:
# Build FAISS index once from chunk embeddings.
chunk_texts = [c.text for c in chunks]
embeddings = embed_texts(chunk_texts)
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
print(f"FAISS vectors: {index.ntotal}")

# Persist index + metadata so the query service does not recompute embeddings on every restart.
faiss.write_index(index, str(ARTIFACT_DIR / "scheme.index"))
with open(ARTIFACT_DIR / "chunks.json", "w", encoding="utf-8") as f:
    json.dump([asdict(c) for c in chunks], f, ensure_ascii=False, indent=2)
print("Saved RAG artifacts")

In [ ]:
def retrieve_documents(
    query: str,
    top_k: int = DEFAULT_TOP_K,
    min_score: float | None = DEFAULT_MIN_SCORE,
) -> list[RetrievalResult]:
    query = clean_text(query)
    if not query:
        return []
    q = embed_texts([query])
    scores, indices = index.search(q, top_k)
    results: list[RetrievalResult] = []
    for score, idx in zip(scores[0], indices[0]):
        if idx < 0:
            continue
        if min_score is not None and float(score) < min_score:
            continue
        c = chunks[int(idx)]
        results.append(RetrievalResult(
            text=c.text, source=c.source, page=c.page, chunk_id=c.chunk_id, score=float(score)
        ))
    return results

def build_context(results: list[RetrievalResult]) -> str:
    blocks = []
    for i, r in enumerate(results, start=1):
        blocks.append(
            f"DOCUMENT {i}\nSOURCE: {r.source}\nPAGE: {r.page}\nCONTENT: {r.text}"
        )
    return "\n\n".join(blocks)

In [ ]:
SYSTEM_PROMPT = """You are ArogyaVani, a trusted government health-scheme information assistant.
Answer using ONLY the supplied trusted document context.
Do not invent facts, links, eligibility, benefits, documents, or application steps.
Do not use outside knowledge.
Do not diagnose or prescribe medicines.
Keep answers simple, concise, and practical.
If the context does not contain enough information, clearly say that the trusted documents do not provide the requested information.
Use the user's requested language when instructed.
"""

def generate_answer(question: str, context: str, language_code: str = "en-IN") -> str:
    if not GEMINI_API_KEY:
        raise RuntimeError("GEMINI_API_KEY is not set for a live Gemini run.")
    client = genai.Client(api_key=GEMINI_API_KEY)
    prompt = f"""{SYSTEM_PROMPT}

USER LANGUAGE: {language_code}
USER QUESTION: {question}

TRUSTED DOCUMENTS:
{context}

ANSWER:"""
    response = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=prompt,
    )
    answer = (response.text or "").strip()
    if not answer:
        raise RuntimeError("Gemini returned an empty answer.")
    return answer

In [ ]:
def rag_answer(
    question: str,
    language_code: str = "en-IN",
    top_k: int = DEFAULT_TOP_K,
    min_score: float | None = DEFAULT_MIN_SCORE,
) -> dict[str, Any]:
    retrieved = retrieve_documents(question, top_k=top_k, min_score=min_score)
    if not retrieved:
        return {
            "answer": "I could not find relevant information in the trusted scheme documents.",
            "sources": [],
        }

    answer = generate_answer(question, build_context(retrieved), language_code)
    seen: set[tuple[str, int]] = set()
    sources: list[dict[str, Any]] = []
    for r in retrieved:
        key = (r.source, r.page)
        if key in seen:
            continue
        sources.append({"source": r.source, "page": r.page, "score": round(r.score, 4)})
        seen.add(key)
    return {"answer": answer, "sources": sources}

print("Query pipeline ready")

## Scheme-level result preparation

The original notebook returns retrieved chunks and source metadata. For the app UI, keep **scheme identity/action data separate from RAG text retrieval**. RAG should explain and rank relevant documents; a curated scheme catalog should own verified application/eligibility URLs.

This avoids asking Gemini to invent links.

In [ ]:
# Future-ready catalog shape; populate only from verified official URLs/data.
SCHEME_CATALOG_EXAMPLE = {
    "pmmvy": {
        "name": "Pradhan Mantri Matru Vandana Yojana",
        "action_type": "apply",
        "official_website": None,
        "application_url": None,
    }
}

# Helper to map source filenames to scheme IDs. Keep this mapping curated.
SOURCE_TO_SCHEME = {
    "pmmvy_scheme.pdf": "pmmvy",
    "jsy_scheme.pdf": "jsy",
    "jssk_scheme.pdf": "jssk",
    "pmsma_scheme.pdf": "pmsma",
    "rbsk_scheme.pdf": "rbsk",
    "nhm_scheme.pdf": "nhm",
}

def scheme_candidates(results: list[RetrievalResult]) -> list[dict[str, Any]]:
    grouped: dict[str, list[float]] = {}
    for r in results:
        scheme_id = SOURCE_TO_SCHEME.get(r.source)
        if not scheme_id:
            continue
        grouped.setdefault(scheme_id, []).append(r.score)
    ranked = sorted(
        ((scheme_id, max(scores)) for scheme_id, scores in grouped.items()),
        key=lambda x: x[1],
        reverse=True,
    )
    return [
        {
            "scheme_id": scheme_id,
            "relevance_score": round(score, 4),
            "scheme": SCHEME_CATALOG_EXAMPLE.get(scheme_id),
        }
        for scheme_id, score in ranked
    ]

## Lightweight offline validation

The real embedding/Gemini cells require third-party packages, model downloads, and an API key. This notebook includes a dependency-free structural test so the architecture can be validated without credentials.

In [ ]:
# Offline structural checks using deterministic fake vectors.
class FakeEmbedder:
    dim = 8
    def encode(self, texts, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False):
        out = []
        for t in texts:
            seed = sum(ord(ch) for ch in t) % 997
            rng = np.random.default_rng(seed)
            v = rng.normal(size=self.dim).astype(np.float32)
            v /= np.linalg.norm(v)
            out.append(v)
        return np.stack(out)

fake = FakeEmbedder()
test_docs = [
    PageDocument("JSSK free and cashless delivery for pregnant women in government facilities.", "jssk_scheme.pdf", 1),
    PageDocument("PMMVY provides maternity cash benefits subject to eligibility requirements.", "pmmvy_scheme.pdf", 1),
]
test_chunks = build_chunks(test_docs)
assert len(test_chunks) == 2
assert all(c.source.endswith(".pdf") for c in test_chunks)
assert all(c.page == 1 for c in test_chunks)
print("✅ Offline ingestion/chunk metadata test passed")
print("✅ Source/page metadata preserved")
print("✅ Scheme catalog separation prepared")

In [ ]:
# Optional real RAG smoke test.
# Set GEMINI_API_KEY and place PDFs under trusted_pdfs/ before running this cell.
if documents and chunks and GEMINI_API_KEY:
    result = rag_answer("Who can benefit from JSSK?", language_code="en-IN", top_k=5)
    print(json.dumps(result, indent=2, ensure_ascii=False))
else:
    print("Skipped live RAG smoke test: missing PDFs and/or GEMINI_API_KEY.")

## Production handoff

**Do not deploy this notebook as the backend.** Convert the reusable functions into a small FastAPI RAG service/module.

Recommended production assets:

```text
backend/
├── services/
│   ├── rag_service.py
│   ├── rag_ingestion.py
│   └── ...
├── rag_artifacts/
│   ├── scheme.index
│   └── chunks.json
└── ...
```

The existing voice backend should call the RAG service after STT/translation for scheme questions, then pass the final text into the existing TTS path.